# Voice Cloning & Text-to-Speech Pipeline
## Using Edge-TTS + RVC (Retrieval-based Voice Conversion)

This notebook demonstrates a complete pipeline to:
1. Convert text to speech using Edge-TTS (base audio - "skeleton")
2. Transform that speech to sound like your voice using an RVC model trained with Applio

**Prerequisites:**
- RVC model files (.pth and .index) trained externally (e.g., via Applio)
- GPU recommended for faster voice conversion
- Transcript text (from previous cells or transcript.txt file)

## Step 1: Install Required Dependencies
Run this cell first to install the necessary packages.

In [5]:
!pip install git+https://github.com/daswer123/rvc-python.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/daswer123/rvc-python.git to c:\users\rohan\appdata\local\temp\pip-req-build-17ze77rd
  Resolved https://github.com/daswer123/rvc-python.git to commit cff3ffbe99ce2e062831e4377f8cef587d0f38f9
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached av-16.1.0-cp313-cp313-win_amd64.whl.metadata (4.7 kB)
  Using cached fairseq-0.12.2.tar.gz (9.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: starte

  Running command git clone --filter=blob:none --quiet https://github.com/daswer123/rvc-python.git 'C:\Users\rohan\AppData\Local\Temp\pip-req-build-17ze77rd'
ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.3 (from rvc-python) (from versions: 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2)
ERROR: No matching distribution found for faiss-cpu==1.7.3


In [6]:
# Install required packages
# Note: Run this only once or if packages are not installed

# Step 1: Upgrade pip, setuptools, and wheel (important for Python 3.13+)
print("📦 Upgrading pip, setuptools, and wheel...")
!pip install --upgrade pip setuptools wheel

# Step 2: Install packages individually
print("\n📥 Installing edge-tts...")
!pip install edge-tts

print("\n📥 Installing PyTorch (CPU version)...")
!pip install torch torchaudio

print("\n📥 Installing audio processing libraries...")
!pip install librosa soundfile scipy

print("\n📥 Installing rvc-python (this may take a moment)...")
!pip install rvc-python

print("\n🔍 Verifying installation...")
try:
    import edge_tts
    print("✅ edge-tts installed")
except:
    print("❌ edge-tts failed")

try:
    import torch
    print("✅ torch installed")
except:
    print("❌ torch failed")

try:
    from rvc_python.infer import RVCInference
    print("✅ rvc-python installed")
except Exception as e:
    print(f"❌ rvc-python failed: {e}")
    print("\n⚠️  If rvc-python fails, you may need to:")
    print("   1. Use Python 3.11 instead of 3.13")
    print("   2. Or install from source: pip install git+https://github.com/daswer123/rvc-python.git")

print("\n✅ Installation complete!")

📦 Upgrading pip, setuptools, and wheel...
Defaulting to user installation because normal site-packages is not writeable

📥 Installing edge-tts...
Defaulting to user installation because normal site-packages is not writeable

📥 Installing PyTorch (CPU version)...
Defaulting to user installation because normal site-packages is not writeable

📥 Installing audio processing libraries...
Defaulting to user installation because normal site-packages is not writeable

📥 Installing rvc-python (this may take a moment)...
Defaulting to user installation because normal site-packages is not writeable
  Using cached rvc_python-0.1.5-py3-none-any.whl.metadata (8.6 kB)
  Using cached av-16.1.0-cp313-cp313-win_amd64.whl.metadata (4.7 kB)
  Using cached fairseq-0.12.2.tar.gz (9.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Instal

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\rohan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pip\_internal\cli\base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "C:\Users\rohan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pip\_internal\cli\base_command.py", line 98, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\rohan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pip\_internal\cli\req_command.py", line 85, in wrapper
    return func(self, options, args)
  File "C:\Users\rohan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pip\_internal\commands\install.py", line 388, in r

### ⚠️ If rvc-python failed above, run this cell to install from source:

In [7]:
# Alternative: Use pyttsx3 for basic TTS (works with Python 3.13)
# Or use edge-tts only without RVC voice cloning
print("⚠️ rvc-python is incompatible with Python 3.13")
print("\nRecommended solutions:")
print("1. ✅ Use Python 3.11 (create a new environment)")
print("2. ⚠️ Use TTS only without voice cloning (basic pipeline)")
print("\nChoose option 2 for now to test basic TTS functionality...")

⚠️ rvc-python is incompatible with Python 3.13

Recommended solutions:
1. ✅ Use Python 3.11 (create a new environment)
2. ⚠️ Use TTS only without voice cloning (basic pipeline)

Choose option 2 for now to test basic TTS functionality...


## Step 2: Import Libraries and Check Environment

In [8]:
# Import necessary libraries
import os
import edge_tts
from IPython.display import Audio, display
import torch

# RVC is not available with Python 3.13, so we'll use TTS only for now
print("⚠️ Note: Running in TTS-only mode (no voice cloning)")
print("For voice cloning, please use Python 3.11 or earlier\n")

# Check if CUDA (GPU) is available
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")
if device == "cuda:0":
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected.")

⚠️ Note: Running in TTS-only mode (no voice cloning)
For voice cloning, please use Python 3.11 or earlier

🖥️  Using device: cpu
⚠️  No GPU detected.


## Step 3: Configuration - Define Paths
Set up paths for your RVC model files (trained with Applio).

In [9]:
# ========================================
# CONFIGURATION - Update these paths!
# ========================================

# Path to your trained RVC model file (.pth)
MODEL_PATH = "my_voice.pth"

# Path to your RVC index file (.index) - optional but recommended for better quality
INDEX_PATH = "my_voice.index"

# Output file paths
TEMP_BASE_AUDIO = "temp_base.mp3"  # Temporary TTS output
FINAL_OUTPUT_AUDIO = "final_cloned_output.wav"  # Final voice-cloned audio

# Edge-TTS voice configuration
# See full list: https://speech.microsoft.com/portal/voicegallery
TTS_VOICE = "en-US-ChristopherNeural"  # Clear, neutral male voice

print("📁 Configuration:")
print(f"  Model Path: {MODEL_PATH}")
print(f"  Index Path: {INDEX_PATH}")
print(f"  TTS Voice: {TTS_VOICE}")
print(f"  Output: {FINAL_OUTPUT_AUDIO}")

📁 Configuration:
  Model Path: my_voice.pth
  Index Path: my_voice.index
  TTS Voice: en-US-ChristopherNeural
  Output: final_cloned_output.wav


## Step 4: Load or Create Transcript
Check for transcript file or use sample text.

In [10]:
# Load transcript from file or use dummy text
transcript_file = "transcript.txt"

if os.path.exists(transcript_file):
    with open(transcript_file, "r", encoding="utf-8") as f:
        final_transcript = f.read().strip()
    print(f"✅ Loaded transcript from '{transcript_file}'")
else:
    # Dummy transcript for testing
    final_transcript = """
    Hello! This is a demonstration of voice cloning technology using RVC models.
    The text you're hearing was first converted to speech using Microsoft Edge TTS,
    and then transformed to sound like a specific person's voice using a trained RVC model.
    This powerful pipeline enables high-quality voice synthesis with natural-sounding results.
    """
    print(f"⚠️  File '{transcript_file}' not found. Using sample transcript.")

print(f"\n📝 Transcript preview ({len(final_transcript)} characters):")
print(final_transcript[:200] + "..." if len(final_transcript) > 200 else final_transcript)

⚠️  File 'transcript.txt' not found. Using sample transcript.

📝 Transcript preview (357 characters):

    Hello! This is a demonstration of voice cloning technology using RVC models.
    The text you're hearing was first converted to speech using Microsoft Edge TTS,
    and then transformed to sound ...


## Step 5: Generate Base Audio with Edge-TTS
This creates the "skeleton" audio using Microsoft Edge TTS.

In [11]:
async def generate_base_audio(text, output_filename):
    """
    Generate base audio from text using Edge-TTS.
    
    Args:
        text (str): The text to convert to speech
        output_filename (str): Path to save the audio file
    
    Returns:
        str: Path to the generated audio file
    """
    print(f"🎤 Generating base audio with {TTS_VOICE}...")
    
    # Create TTS communication object
    communicate = edge_tts.Communicate(text, TTS_VOICE)
    
    # Save audio to file
    await communicate.save(output_filename)
    
    print(f"✅ Base audio saved to: {output_filename}")
    return output_filename

## Step 6: Initialize RVC Model
Verify model files and set up the RVC inference engine.

In [12]:
# Skip RVC initialization for now (Python 3.13 compatibility issue)
print("⚠️ RVC voice cloning is disabled (Python 3.13 incompatibility)")
print("This notebook will generate TTS audio only.\n")
print("To enable voice cloning:")
print("1. Create a Python 3.11 virtual environment:")
print("   python -m venv venv_py311 --python=3.11")
print("2. Activate it and reinstall packages")
print("3. Re-run this notebook")

# Set flag to skip voice cloning
SKIP_VOICE_CLONING = True
print("\n✅ Continuing with TTS-only mode...")

⚠️ RVC voice cloning is disabled (Python 3.13 incompatibility)
This notebook will generate TTS audio only.

To enable voice cloning:
1. Create a Python 3.11 virtual environment:
   python -m venv venv_py311 --python=3.11
2. Activate it and reinstall packages
3. Re-run this notebook

✅ Continuing with TTS-only mode...


## Step 7: Convert Audio to Cloned Voice
Apply voice conversion using your RVC model.

In [13]:
# This function is disabled in TTS-only mode
def convert_to_my_voice(input_audio, output_audio):
    """
    Placeholder function - voice conversion not available with Python 3.13.
    
    To enable voice cloning, use Python 3.11 or earlier.
    """
    print(f"\n⚠️ Voice conversion skipped (Python 3.13 incompatibility)")
    print(f"Using TTS audio without voice cloning: {input_audio}")
    
    # Just copy the input to output
    import shutil
    shutil.copy(input_audio, output_audio)
    
    return output_audio

## Step 8: Execute the Complete Pipeline
Run the full TTS → Voice Cloning pipeline and play the result.

In [14]:
# ========================================
# MAIN EXECUTION PIPELINE (TTS-ONLY MODE)
# ========================================

print("=" * 60)
print("🚀 Starting Text-to-Speech Pipeline (No Voice Cloning)")
print("=" * 60)

try:
    # Step 1: Generate base audio using Edge-TTS
    await generate_base_audio(final_transcript, TEMP_BASE_AUDIO)
    
    # Step 2: In TTS-only mode, just rename the output
    print("\n⚠️ Voice cloning skipped - using TTS audio directly")
    import shutil
    shutil.copy(TEMP_BASE_AUDIO, FINAL_OUTPUT_AUDIO)
    
    # Step 3: Display and play the final audio
    print("\n" + "=" * 60)
    print("🎉 SUCCESS! TTS audio generated!")
    print("=" * 60)
    print(f"📁 Final output saved to: {FINAL_OUTPUT_AUDIO}")
    print("\n💡 To enable voice cloning, use Python 3.11 or earlier")
    
    # Play audio in notebook with autoplay
    display(Audio(FINAL_OUTPUT_AUDIO, autoplay=True))
    
    # Step 4: Clean up temporary files
    if os.path.exists(TEMP_BASE_AUDIO) and TEMP_BASE_AUDIO != FINAL_OUTPUT_AUDIO:
        os.remove(TEMP_BASE_AUDIO)
        print(f"🧹 Cleaned up temporary file: {TEMP_BASE_AUDIO}")
    
except FileNotFoundError as e:
    print(f"\n❌ Error: {e}")
except Exception as e:
    print(f"\n❌ Unexpected error: {e}")
    import traceback
    traceback.print_exc()
finally:
    print("\n✅ Pipeline execution complete.")

🚀 Starting Text-to-Speech Pipeline (No Voice Cloning)
🎤 Generating base audio with en-US-ChristopherNeural...
✅ Base audio saved to: temp_base.mp3

⚠️ Voice cloning skipped - using TTS audio directly

🎉 SUCCESS! TTS audio generated!
📁 Final output saved to: final_cloned_output.wav

💡 To enable voice cloning, use Python 3.11 or earlier


🧹 Cleaned up temporary file: temp_base.mp3

✅ Pipeline execution complete.


---

## 📝 Usage Instructions

1. **Train Your Voice Model**: Use [Applio](https://github.com/IAHispano/Applio) to train an RVC model with your voice samples
2. **Place Model Files**: Copy `my_voice.pth` (and optionally `my_voice.index`) to this directory
3. **Update Configuration**: Modify the paths in Step 3 if your files have different names
4. **Run All Cells**: Execute cells sequentially from top to bottom
5. **Listen**: The final audio will auto-play when complete!

## 🎛️ Parameter Tuning Guide

- **`f0_method`**: Pitch extraction method
  - `"rmvpe"` (recommended): Highest quality, slower
  - `"harvest"`: Good quality, faster
  - `"crepe"`: Very accurate, slowest

- **`index_rate`** (0.0 - 1.0): How much to preserve voice characteristics
  - `0.0`: More like base voice
  - `0.75` (recommended): Balanced
  - `1.0`: Maximum voice preservation

- **`protect`** (0.0 - 0.5): Protects consonants from artifacts
  - `0.0`: No protection
  - `0.33` (recommended): Balanced
  - `0.5`: Maximum protection

## 🔧 Troubleshooting

- **"Model file not found"**: Train a model with Applio and place it in this directory
- **Slow conversion**: Use a GPU or try `f0_method="harvest"`
- **Robotic sound**: Increase `protect` value (try 0.4-0.5)
- **Not enough voice resemblance**: Increase `index_rate` (try 0.85-0.95)